# TỰ ĐỘNG NHẬN DIỆN VÀ ĐỌC ĐA ĐỊNH DẠNG FILE BÁO CÁO TMĐT BẨN

In [1]:
import os
import sys
import joblib
import pandas as pd
import numpy as np

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
MESSY_DIR = os.path.join(ROOT, 'data', 'samples_messy')
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(ROOT, 'models')

In [3]:
#Xem danh sách 10 file xuất thô mô phỏng các sàn TMĐT
files = sorted(os.listdir(MESSY_DIR))
print('Số lượng file test:', len(files))
files

Số lượng file test: 10


['01_shopee_clean.xlsx',
 '02_tiki_semicolon_bom.csv',
 '03_lazada_english.xlsx',
 '04_tiktok_title_offset.xlsx',
 '05_pos_legacy.xls',
 '06_multi_sheet_report.xlsx',
 '07_utf16_tab_export.csv',
 '08_two_text_cols.xlsx',
 '09_merged_cells_pagination.xlsx',
 '10_html_table_as_xls.xls']

## 1. ĐỌC ĐA ĐỊNH DẠNG FILE BÁO CÁO

In [4]:
#Nạp bộ đọc và nhận diện cấu trúc bảng Excel/CSV dùng chung từ src.excel_ingestion
from src.excel_ingestion import (
    read_any_file, pick_best_sheet, detect_header_row,
    detect_text_columns, detect_star_column, build_clean_table, mask_tabular_pii
)

In [5]:
#Thử nghiệm đọc file HTML đội lốt XLS bằng read_any_file
sheets_10, meta_10 = read_any_file(os.path.join(MESSY_DIR, '10_html_table_as_xls.xls'))
tempt_df = sheets_10['Sheet1']
print('Định dạng nhận diện:', meta_10['format'])
tempt_df.head(2)

Định dạng nhận diện: html_xls


,Mã ĐH,Người Mua,Nội Dung Đánh Giá,Đánh Giá Sao,Thời Gian
0,ORD933552,Hoàng Thảo Vy,"Sản phẩm tốt, nhưng pin không trâu như mình ng...",3,2026-10-09
1,ORD479410,Đỗ Minh Đức,Hàng chất lượng sai rất ok cho thế giới đi đôn...,5,2026-10-15


In [6]:
tempt_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Mã ĐH              25 non-null     str  
 1   Người Mua          25 non-null     str  
 2   Nội Dung Đánh Giá  25 non-null     str  
 3   Đánh Giá Sao       25 non-null     int64
 4   Thời Gian          25 non-null     str  
dtypes: int64(1), str(4)
memory usage: 6.8 KB


## 2. TỰ ĐỘNG CHỌN SHEET CHỨA NHẬN XÉT

In [7]:
#Kiểm tra tự chọn sheet trên file nhiều sheet 06_multi_sheet_report.xlsx
sheets_06, meta_06 = read_any_file(os.path.join(MESSY_DIR, '06_multi_sheet_report.xlsx'))
best_sname, df_best = pick_best_sheet(sheets_06)
print('Sheet được chọn tự động:', best_sname)
df_best.head(2)

Sheet được chọn tự động: Danh_Gia_Chi_Tiet


,0,1,2,3,4
0,Mã Đơn,Khách Hàng,Nội dung phản hồi,Đánh giá sao,Chi nhánh
1,ORD782784,Hoàng Thảo Vy,"Sản phẩm tốt, nhưng pin không trâu như mình ng...",3,Hà Nội


## 3. TỰ ĐỘNG DÒ DÒNG TIÊU ĐỀ VÀ CỘT ĐÁNH GIÁ

In [8]:
#Thử dò dòng tiêu đề thật khi bảng bị đẩy lệch bởi banner trên file 04
sheets_04, meta_04 = read_any_file(os.path.join(MESSY_DIR, '04_tiktok_title_offset.xlsx'))
sname_04, raw_04 = pick_best_sheet(sheets_04)
hdr_04 = detect_header_row(raw_04)
print('Dòng tiêu đề nhận diện được:', hdr_04)

Dòng tiêu đề nhận diện được: 4


In [9]:
#Kiểm tra dò cột văn bản và cột số sao trên file 04 sau khi làm sạch bảng
clean_tab_04 = build_clean_table(raw_04, hdr_04, meta=meta_04, sheet_name=sname_04)
detect_text_columns(clean_tab_04), detect_star_column(clean_tab_04)

(['Ý kiến đánh giá', 'review_text'], 'Số sao')

## 4. XỬ LÝ LẶP TIÊU ĐỀ, Ô GỘP VÀ ĐA CỘT VĂN BẢN

In [10]:
#Thử nghiệm gộp 2 cột nhận xét thành review_text trên file 08_two_text_cols.xlsx
sheets_08, meta_08 = read_any_file(os.path.join(MESSY_DIR, '08_two_text_cols.xlsx'))
sname_08, raw_08 = pick_best_sheet(sheets_08)
clean_tab_08 = build_clean_table(raw_08, detect_header_row(raw_08), meta=meta_08, sheet_name=sname_08)
clean_tab_08[['Nhận xét về máy', 'Ý kiến đóng gói giao hàng', 'review_text']].head(3)

,Nhận xét về máy,Ý kiến đóng gói giao hàng,review_text
0,"Sản phẩm tốt, nhưng pin không trâu như mình ng...",Hộp đóng gói cẩn thận 3 lớp bóng khí bọc quanh...,"Sản phẩm tốt, nhưng pin không trâu như mình ng..."
1,Hàng chất lượng sai rất ok cho thế giới đi đôn...,Giao hơi chậm 1 ngày so với dự kiến nhưng ship...,Hàng chất lượng sai rất ok cho thế giới đi đôn...
2,E mới mưa đc 2 tháng. Mà e mới cập nhật phiên ...,Hộp đóng gói cẩn thận 3 lớp bóng khí bọc quanh...,E mới mưa đc 2 tháng. Mà e mới cập nhật phiên ...


In [11]:
clean_tab_08.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Mã đơn                     25 non-null     str   
 1   Khách hàng                 25 non-null     str   
 2   Nhận xét về máy            25 non-null     str   
 3   Ý kiến đóng gói giao hàng  25 non-null     str   
 4   Sao                        25 non-null     object
 5   Ngày                       25 non-null     str   
 6   review_text                25 non-null     str   
dtypes: object(1), str(6)
memory usage: 15.8+ KB


In [12]:
#Truyền meta và sheet_name để chỉ unmerge đúng vùng ô gộp Nhóm sản phẩm và lọc tiêu đề lặp giữa trang
sheets_09, meta_09 = read_any_file(os.path.join(MESSY_DIR, '09_merged_cells_pagination.xlsx'))
sname_09, raw_09 = pick_best_sheet(sheets_09)
clean_tab_09 = build_clean_table(raw_09, detect_header_row(raw_09), meta=meta_09, sheet_name=sname_09)
clean_tab_09.head(3)

,STT,Nhóm sản phẩm,Khách hàng,Phản hồi sản phẩm,Số sao đánh giá,Ghi chú,review_text
0,1,Điện thoại Flagship,Hoàng Thảo Vy,"Sản phẩm tốt, nhưng pin không trâu như mình ng...",3,Đã xử lý,"Sản phẩm tốt, nhưng pin không trâu như mình ng..."
1,2,Điện thoại Flagship,Trần Thị Bích,Hàng chất lượng sai rất ok cho thế giới đi đôn...,5,Chưa xử lý,Hàng chất lượng sai rất ok cho thế giới đi đôn...
2,3,Điện thoại Flagship,Trần Thị Bích,E mới mưa đc 2 tháng. Mà e mới cập nhật phiên ...,4,Chưa xử lý,E mới mưa đc 2 tháng. Mà e mới cập nhật phiên ...


In [13]:
clean_tab_09.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   STT                25 non-null     object
 1   Nhóm sản phẩm      25 non-null     str   
 2   Khách hàng         25 non-null     str   
 3   Phản hồi sản phẩm  25 non-null     str   
 4   Số sao đánh giá    25 non-null     object
 5   Ghi chú            25 non-null     str   
 6   review_text        25 non-null     str   
dtypes: object(2), str(5)
memory usage: 12.6+ KB


In [14]:
#Xác nhận file 09 còn đúng 25 dòng, 0 NaN ở Nhóm sản phẩm và không bị ffill bừa
print('Số dòng:', len(clean_tab_09), '| NaN Nhóm sản phẩm:', clean_tab_09['Nhóm sản phẩm'].isna().sum())
clean_tab_09['Nhóm sản phẩm'].value_counts()

Số dòng: 25 | NaN Nhóm sản phẩm: 0


Nhóm sản phẩm
Điện thoại Flagship     5
Điện thoại Tầm trung    5
Phụ kiện & Sạc nhanh    5
Điện thoại Phổ thông    5
Dịch vụ & Hậu mãi       5
Name: count, dtype: int64

In [15]:
#Kiểm thử nhanh giữ nguyên ô trống ở cột SĐT/Số sao vì chỉ unmerge đúng merged_ranges
tempt_raw = pd.DataFrame([['Nhóm', 'SĐT', 'Nhận xét', 'Số sao'], ['A', '0901234567', 'Sản phẩm tốt lắm ạ', 5], [None, None, 'Pin hơi yếu một chút', None]])
tempt_res = build_clean_table(tempt_raw, header_row=0, meta={'merged_ranges': {'S1': [(2, 3, 1, 1)]}}, sheet_name='S1')
assert tempt_res['Nhóm'].tolist() == ['A', 'A'] and pd.isna(tempt_res.loc[1, 'SĐT']) and pd.isna(tempt_res.loc[1, 'Số sao'])
tempt_res

,Nhóm,SĐT,Nhận xét,Số sao,review_text
0,A,0901234567,Sản phẩm tốt lắm ạ,5,Sản phẩm tốt lắm ạ
1,A,NaN,Pin hơi yếu một chút,None,Pin hơi yếu một chút


## 5. CHẠY KIỂM ĐỊNH BENCHMARK TRÊN TOÀN BỘ 10 FILE BẨN

In [16]:
#Thiết lập nhãn cột chuẩn cho 10 file xuất thô
truth = {
    '01_shopee_clean.xlsx': {'text': ['Đánh giá của khách'], 'star': 'Số sao (1-5)'}, '02_tiki_semicolon_bom.csv': {'text': ['Review_Content'], 'star': 'Rating_Star'},
    '03_lazada_english.xlsx': {'text': ['Comment / Review'], 'star': 'Rating'}, '04_tiktok_title_offset.xlsx': {'text': ['Ý kiến đánh giá'], 'star': 'Số sao'},
    '05_pos_legacy.xls': {'text': ['Nhan_Xet'], 'star': 'Diem_Sao'}, '06_multi_sheet_report.xlsx': {'text': ['Nội dung phản hồi'], 'star': 'Đánh giá sao'},
    '07_utf16_tab_export.csv': {'text': ['Review_Text'], 'star': 'Stars'}, '08_two_text_cols.xlsx': {'text': ['Nhận xét về máy', 'Ý kiến đóng gói giao hàng'], 'star': 'Sao'},
    '09_merged_cells_pagination.xlsx': {'text': ['Phản hồi sản phẩm'], 'star': 'Số sao đánh giá'}, '10_html_table_as_xls.xls': {'text': ['Nội Dung Đánh Giá'], 'star': 'Đánh Giá Sao'}
}

In [17]:
#Chạy kiểm định tự động trên cả 10 file bẩn
results = []
for fname, expected in sorted(truth.items()):
    sheets, meta = read_any_file(os.path.join(MESSY_DIR, fname))
    best_sname, best_df = pick_best_sheet(sheets)
    tab = build_clean_table(best_df, detect_header_row(best_df), meta=meta, sheet_name=best_sname)
    det_txt, det_star = detect_text_columns(tab), detect_star_column(tab)
    ok = (len(tab) == 25) and all(t in det_txt for t in expected['text']) and (det_star == expected['star']) and bool((tab['review_text'].str.len() > 0).all())
    results.append({'file_name': fname, 'rows': len(tab), 'format': meta['format'], 'detected_text': ', '.join(det_txt), 'detected_star': str(det_star), 'pass_status': 'PASS' if ok else 'FAIL'})
df_benchmark = pd.DataFrame(results)

In [18]:
print('Tỷ lệ nhận diện thành công: ', (df_benchmark['pass_status'] == 'PASS').sum(), '/', len(df_benchmark))
df_benchmark

Tỷ lệ nhận diện thành công:  10 / 10


,file_name,rows,format,detected_text,detected_star,pass_status
0,01_shopee_clean.xlsx,25,xlsx,"Đánh giá của khách, review_text",Số sao (1-5),PASS
1,02_tiki_semicolon_bom.csv,25,csv,"Review_Content, review_text",Rating_Star,PASS
2,03_lazada_english.xlsx,25,xlsx,"Comment / Review, review_text",Rating,PASS
3,04_tiktok_title_offset.xlsx,25,xlsx,"Ý kiến đánh giá, review_text",Số sao,PASS
4,05_pos_legacy.xls,25,biff_xls,"Nhan_Xet, review_text",Diem_Sao,PASS
5,06_multi_sheet_report.xlsx,25,xlsx,"Nội dung phản hồi, review_text",Đánh giá sao,PASS
6,07_utf16_tab_export.csv,25,csv,"Review_Text, review_text",Stars,PASS
7,08_two_text_cols.xlsx,25,xlsx,"Nhận xét về máy, Ý kiến đóng gói giao hàng, re...",Sao,PASS
8,09_merged_cells_pagination.xlsx,25,xlsx,"Phản hồi sản phẩm, review_text",Số sao đánh giá,PASS
9,10_html_table_as_xls.xls,25,html_xls,"Nội Dung Đánh Giá, review_text",Đánh Giá Sao,PASS


## 6. CHUẨN HÓA VĂN BẢN, ẨN DANH PII VÀ TÁCH TỪ PYVI TRƯỚC DỰ ĐOÁN

In [19]:
#Nạp bộ làm sạch văn bản, lọc câu rác và tách từ pyvi dùng chung từ src.text_cleaner
from src.text_cleaner import clean_text, check_no_content, segment_words

In [20]:
#Thử nghiệm làm sạch, lọc nội dung và tách từ trên 1 câu mẫu chứa SĐT và mã đơn
tempt = 'Đơn ORD967144 của sđt 0779887766: TGDĐ đt dùng rất mượttt nhưng pin tụt ko dùng đk lâu'
tempt_clean = clean_text(tempt, lowercase=False)
tempt_seg = segment_words(tempt_clean)
tempt_clean, tempt_seg, check_no_content(tempt_clean), check_no_content('cho 5 sao nhận xu')

('Đơn [ORDER_ID] của sđt [PHONE] : Thế Giới Di Động điện thoại dùng rất mượt nhưng pin tụt không dùng được lâu',
 'Đơn [ORDER_ID] của sđt [PHONE] : Thế_Giới_Di_Động điện_thoại dùng rất mượt nhưng pin tụt không dùng được lâu',
 False,
 True)

## 7. DỰ ĐOÁN HÀNG LOẠT VÀ XUẤT FILE SAMPLE_EXCEL_PREDICTED.XLSX

In [21]:
import json, time, warnings
import torch
from transformers import AutoConfig, AutoTokenizer
from src.labels import (
    ABSA_COLS, ABSA_STATE_VI, ASPECT_GROUPS,
    classify_silent_complaint_levels, derive_overall_sentiment,
)
from src.modeling import ABSATemperatureScaler, MAX_LEN_ABSA, PhoBertABSA

#Nạp cả Baseline 5x4 (fallback nhanh) và PhoBERT ABSA v4.0 + Dynamic CPU Speed Router (có warm-up)
tfidf = joblib.load(os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))
base_absa_bundle = joblib.load(os.path.join(MODEL_DIR, 'baseline_absa_5x4.pkl'))
SAVE_DIR = os.path.join(MODEL_DIR, 'phobert_absa_v4')
weights_p = os.path.join(SAVE_DIR, 'phobert_absa_best.pt')
calib_p = os.path.join(SAVE_DIR, 'calibration.json')

phobert_bundle = None
if os.path.exists(weights_p) and os.path.exists(calib_p):
    tok = AutoTokenizer.from_pretrained(SAVE_DIR)
    cfg = AutoConfig.from_pretrained(SAVE_DIR)
    cfg.vocab_size = len(tok)
    with torch.device('meta'):
        m = PhoBertABSA(config=cfg, dropout=0.2)
    m.load_state_dict(torch.load(weights_p, map_location='cpu', mmap=True, weights_only=True), assign=True)
    m.encoder.embeddings.position_ids = torch.arange(cfg.max_position_embeddings, dtype=torch.long).expand((1, -1))
    m.encoder.embeddings.token_type_ids = torch.zeros(m.encoder.embeddings.position_ids.size(), dtype=torch.long)
    m.eval()
    c_info = json.load(open(calib_p, encoding='utf-8'))
    sc = ABSATemperatureScaler.from_dict(c_info)
    # Warm-up 1 pass trước khi bấm giờ tốc độ CPU
    w_enc = tok(['Máy dùng tốt pin khoẻ'], padding=True, truncation=True, max_length=MAX_LEN_ABSA, return_tensors='pt')
    with torch.no_grad():
        _ = m(w_enc['input_ids'], w_enc['attention_mask'])
    phobert_bundle = {'model': m, 'tokenizer': tok, 'scaler': sc, 'review_conf_threshold': c_info['review_conf_threshold']}
print('Loaded baseline family:', base_absa_bundle['family'], '| PhoBERT available:', phobert_bundle is not None)


Loaded baseline family: LinearSVC | PhoBERT available: True


In [22]:
ASPECT_COL_VI = [
    'Hieu_Nang_Cau_Hinh',
    'Man_Hinh_Camera',
    'Pin_Sac',
    'Gia_Ca_Hau_Mai',
    'Thiet_Ke_Tong_Quan',
]
SENT_MAP_VI = {0: 'Tiêu cực', 1: 'Trung lập', 2: 'Tích cực'}

def predict_excel_absa_router(df_in: pd.DataFrame, star_col, phobert_bd, base_bd, tfidf_vec, max_wait_sec: float = 60.0, force_fallback: bool = False) -> pd.DataFrame:
    df_out = df_in.copy()
    seg_texts = df_out['sentence_seg'].fillna('').astype(str).tolist()
    n_rows = max(len(seg_texts), 1)
    use_phobert = False
    sec_per_row = None

    if phobert_bd is not None and not force_fallback:
        probe = seg_texts[:min(8, n_rows)]
        t0 = time.perf_counter()
        enc_p = phobert_bd['tokenizer'](probe, padding=True, truncation=True, max_length=MAX_LEN_ABSA, return_tensors='pt')
        with torch.no_grad():
            _ = phobert_bd['model'](enc_p['input_ids'], enc_p['attention_mask'])
        sec_per_row = (time.perf_counter() - t0) / len(probe)
        est_total = sec_per_row * n_rows
        print(f'CPU benchmark (after warm-up): {sec_per_row*1000:.1f} ms/row | Est {n_rows} rows: {est_total:.2f}s (limit {max_wait_sec}s)')
        if est_total <= max_wait_sec:
            use_phobert = True

    if use_phobert:
        model_name = 'PhoBERT_ABSA_v4'
        all_logits = []
        for start in range(0, len(seg_texts), 32):
            enc = phobert_bd['tokenizer'](seg_texts[start:start+32], padding=True, truncation=True, max_length=MAX_LEN_ABSA, return_tensors='pt')
            with torch.no_grad():
                all_logits.append(phobert_bd['model'](enc['input_ids'], enc['attention_mask']).numpy())
        probs = phobert_bd['scaler'].predict_probs(np.concatenate(all_logits, axis=0))
        pred_absa = probs.argmax(axis=-1)
        min_conf = np.round(probs.max(axis=-1).min(axis=-1), 4)
        low_conf_mask = min_conf < phobert_bd['review_conf_threshold']
        aspect_conf = probs.max(axis=-1)
        asp_th = phobert_bd['review_conf_threshold']
    else:
        fam = base_bd['family']
        model_name = f'Baseline_TFIDF_{fam}'
        warnings.warn(f'CANH BAO FALLBACK: Dang dung {model_name} thay vi PhoBERT_ABSA_v4 (force_fallback={force_fallback})')
        print(f'CANH BAO FALLBACK: Dang dung {model_name} thay vi PhoBERT_ABSA_v4')
        X_tf = tfidf_vec.transform(seg_texts)
        # Tuyet doi khong dat nguong xac suat 0.5 tren decision_function cua LinearSVC
        pred_absa = np.column_stack([m.predict(X_tf) for m in base_bd['models']])
        if all(hasattr(m, 'predict_proba') for m in base_bd['models']):
            pr = np.stack([m.predict_proba(X_tf).max(axis=-1) for m in base_bd['models']], axis=-1)
            min_conf = np.round(pr.min(axis=-1), 4)
            low_conf_mask = min_conf < 0.55
            aspect_conf, asp_th = pr, 0.55
        else:
            min_conf = np.full(len(seg_texts), np.nan)
            aspect_conf, asp_th = None, 0.0
            low_conf_mask = np.zeros(len(seg_texts), dtype=bool)

    for j, col_vi in enumerate(ASPECT_COL_VI):
        df_out[f'pred_absa_{j}'] = pred_absa[:, j]
        df_out[col_vi] = [ABSA_STATE_VI[int(v)] for v in pred_absa[:, j]]
    p_sent = derive_overall_sentiment(pred_absa)
    # 'Hon hop' khi co ca khia canh Tieu cuc lan Tich cuc (vd: 'San pham tot nhung pin khong trau')
    mixed = (pred_absa == 1).any(axis=1) & (pred_absa == 3).any(axis=1)
    df_out['Du_Doan_Cam_Xuc'] = ['Hỗn hợp' if m else SENT_MAP_VI[int(s)] for s, m in zip(p_sent, mixed)]
    df_out['Do_Tu_Tin_Thap_Nhat'] = min_conf
    # Khia canh can kiem: do tin cay thap va du doan khac None (None khong can ai doc lai)
    if aspect_conf is not None:
        df_out['Khia_Canh_Can_Kiem'] = [', '.join(ASPECT_COL_VI[j] for j in range(5) if aspect_conf[i, j] < asp_th and pred_absa[i, j] != 0) for i in range(len(df_out))]
    else:
        df_out['Khia_Canh_Can_Kiem'] = ''
    stars_arr = df_out[star_col].values if (star_col is not None and star_col in df_out.columns) else None
    cnt_neg, levels, is_hidden_high = classify_silent_complaint_levels(stars_arr, pred_absa)
    df_out['So_Khia_Canh_Tieu_Cuc'] = cnt_neg
    df_out['Muc_Do_Khieu_Nai_Ngam'] = levels
    df_out['Khieu_Nai_Ngam_4_5_Sao'] = is_hidden_high
    is_empty = df_out['is_no_content'].astype(bool).values
    if use_phobert:
        df_out['Can_Kiem_Duyet'] = (low_conf_mask | is_empty).astype(bool)
    else:
        df_out['Can_Kiem_Duyet'] = (low_conf_mask | is_empty | is_hidden_high).astype(bool)
    df_out['Model_Su_Dung'] = model_name
    return df_out


In [23]:
#Đọc và chuẩn hóa bảng dữ liệu từ file 01_shopee_clean.xlsx
sheets_sample, meta_sample = read_any_file(os.path.join(MESSY_DIR, '01_shopee_clean.xlsx'))
sname_sample, raw_sample = pick_best_sheet(sheets_sample)
df_sample = build_clean_table(raw_sample, detect_header_row(raw_sample), meta=meta_sample, sheet_name=sname_sample)
txt_cols = [c for c in detect_text_columns(df_sample) if c != 'review_text']
txt_col = txt_cols[0] if txt_cols else 'review_text'
star_col = detect_star_column(df_sample)
df_sample = mask_tabular_pii(df_sample, text_clean_fn=lambda s: clean_text(s, lowercase=False))
df_sample['sentence_clean'] = df_sample['review_text']
df_sample['sentence_seg'] = df_sample['sentence_clean'].apply(segment_words)
df_sample['is_no_content'] = df_sample['sentence_clean'].apply(check_no_content)
df_sample = predict_excel_absa_router(df_sample, star_col, phobert_bundle, base_absa_bundle, tfidf)
df_sample[['Model_Su_Dung', 'Du_Doan_Cam_Xuc', 'So_Khia_Canh_Tieu_Cuc', 'Muc_Do_Khieu_Nai_Ngam', 'Khieu_Nai_Ngam_4_5_Sao', 'Can_Kiem_Duyet'] + ASPECT_COL_VI].head(5)


CPU benchmark (after warm-up): 93.6 ms/row | Est 25 rows: 2.34s (limit 60.0s)


,Model_Su_Dung,Du_Doan_Cam_Xuc,So_Khia_Canh_Tieu_Cuc,Muc_Do_Khieu_Nai_Ngam,Khieu_Nai_Ngam_4_5_Sao,Can_Kiem_Duyet,Hieu_Nang_Cau_Hinh,Man_Hinh_Camera,Pin_Sac,Gia_Ca_Hau_Mai,Thiet_Ke_Tong_Quan
0,PhoBERT_ABSA_v4,Hỗn hợp,1,N/A,False,False,Không có,Không có,Tiêu cực,Không có,Tích cực
1,PhoBERT_ABSA_v4,Tích cực,0,Không,False,False,Không có,Không có,Không có,Tích cực,Tích cực
2,PhoBERT_ABSA_v4,Tiêu cực,1,Cao,True,False,Không có,Không có,Tiêu cực,Không có,Không có
3,PhoBERT_ABSA_v4,Trung lập,0,Không,False,False,Không có,Không có,Không có,Trung lập,Tích cực
4,PhoBERT_ABSA_v4,Tiêu cực,1,N/A,False,False,Tiêu cực,Không có,Không có,Không có,Không có


In [24]:
#Kiểm chứng cả nhánh Fallback LinearSVC và trường hợp file không có cột số sao (star_col=None -> Muc_Do='N/A')
df_fb_test = predict_excel_absa_router(df_sample.head(4), star_col=None, phobert_bd=phobert_bundle, base_bd=base_absa_bundle, tfidf_vec=tfidf, force_fallback=True)
assert (df_fb_test['Muc_Do_Khieu_Nai_Ngam'] == 'N/A').all()
assert (df_fb_test['Model_Su_Dung'] == 'Baseline_TFIDF_LinearSVC').all()
print('Verified LinearSVC fallback & star_col=None -> N/A:', df_fb_test[['Model_Su_Dung', 'Muc_Do_Khieu_Nai_Ngam', 'Khieu_Nai_Ngam_4_5_Sao', 'Can_Kiem_Duyet']].to_dict(orient='records'))


CANH BAO FALLBACK: Dang dung Baseline_TFIDF_LinearSVC thay vi PhoBERT_ABSA_v4
Verified LinearSVC fallback & star_col=None -> N/A: [{'Model_Su_Dung': 'Baseline_TFIDF_LinearSVC', 'Muc_Do_Khieu_Nai_Ngam': 'N/A', 'Khieu_Nai_Ngam_4_5_Sao': False, 'Can_Kiem_Duyet': False}, {'Model_Su_Dung': 'Baseline_TFIDF_LinearSVC', 'Muc_Do_Khieu_Nai_Ngam': 'N/A', 'Khieu_Nai_Ngam_4_5_Sao': False, 'Can_Kiem_Duyet': False}, {'Model_Su_Dung': 'Baseline_TFIDF_LinearSVC', 'Muc_Do_Khieu_Nai_Ngam': 'N/A', 'Khieu_Nai_Ngam_4_5_Sao': False, 'Can_Kiem_Duyet': False}, {'Model_Su_Dung': 'Baseline_TFIDF_LinearSVC', 'Muc_Do_Khieu_Nai_Ngam': 'N/A', 'Khieu_Nai_Ngam_4_5_Sao': False, 'Can_Kiem_Duyet': False}]


C:\Users\Giavy\AppData\Local\Temp\ipykernel_16484\831734106.py:45: UserWarning: CANH BAO FALLBACK: Dang dung Baseline_TFIDF_LinearSVC thay vi PhoBERT_ABSA_v4 (force_fallback=True)
  warnings.warn(f'CANH BAO FALLBACK: Dang dung {model_name} thay vi PhoBERT_ABSA_v4 (force_fallback={force_fallback})')


In [25]:
#Xuất file báo cáo mẫu đã làm sạch, ẩn danh PII và gán nhãn ABSA 5x4
out_pred_excel = os.path.join(PROC_DIR, 'sample_excel_predicted.xlsx')
df_sample.to_excel(out_pred_excel, index=False, engine='openpyxl')
df_check = pd.read_excel(out_pred_excel, keep_default_na=False)
print('Đã xuất thành công:', out_pred_excel, '| shape:', df_check.shape)
print('Muc_Do_Khieu_Nai_Ngam counts:', df_check['Muc_Do_Khieu_Nai_Ngam'].value_counts().to_dict())
if phobert_bundle is not None:
    del phobert_bundle
    import gc; gc.collect()


Đã xuất thành công: D:\E-commerce\data\processed\sample_excel_predicted.xlsx | shape: (25, 28)
Muc_Do_Khieu_Nai_Ngam counts: {'N/A': 10, 'Không': 10, 'Nhẹ': 3, 'Cao': 2}
